PATH PLANNING ALGORITHM BASED ON GENETIC AND HEURISTIC ALGORITHMS

Basic
Operations
for
Rover
Automated
Systems AKA B.O.R.A.S.


In [54]:
import numpy as np
import rasterio as rio
from rasterio.plot import show
import matplotlib.pyplot as plt
from time import perf_counter
import pandas as pd
import elevation 
import random
import copy
import datetime

Function Definitions:

move_position

This function will return the new position after the movement is executed given the following inputs:

- grid_size: This will be in the form of a tuple (x,y) and is used to set the boundaries of the grid.
- position: This will be in the form of a tuple (x,y) and will be used as reference point from where to move from.
- moves: ["U", "D", "L", "R","UL", "DL", "UR","UL"] up & down, left & right and in diagonals.
- end: This will be in the form of a tuple (x,y) and will be used to guarantee that once the final position has been reached there will be no more movements.

In [55]:

def move_position(grid_size,position,move,end):
    x, y = position
    if move == "U" and y > 0:
        new_position = (x,y-1)
    elif move== "D" and y < grid_size[1]-1:
        new_position = (x,y+1)
    elif move == "L" and x > 0:
        new_position = (x-1,y)
    elif move == "R" and x < grid_size[0]-1:
        new_position = (x+1,y)
    elif move == "UL" and y > 0 and x > 0:
        new_position = (x - 1, y - 1)
    elif move == "DL" and y < grid_size[1] - 1 and x > 0:
        new_position = (x - 1, y + 1)
    elif move == "UR" and x < grid_size[0] - 1 and y > 0:
        new_position = (x + 1, y - 1)
    elif move == "DR" and x < grid_size[0] - 1 and y < grid_size[1] - 1:
        new_position = (x + 1, y + 1)
    elif x == end[0] and y == end[1]:
        new_position = position
    else:
        new_position = position
    return new_position

evaluate_move

This function will evaluate the fitness of the suggested move via a greedy policy. The fitness function is the Euclidean distance between the new position and the end position/objective.

- grid_size: This will be in the form of a tuple (x,y) and is used to set the boundaries of the grid.
- position: This will be in the form of a tuple (x,y) and will be used as reference point from where to move from.
- moves: ["U", "D", "L", "R","UL", "DL", "UR","UL"] up & down, left & right and in diagonals.
- start: This will be in the form of a tuple (x,y).
- end: This will be in the form of a tuple (x,y).

In [56]:
def evaluate_move(grid_size,move,start,end,position):
    position = move_position(grid_size,position,move,end)
    distance_to_end=np.sqrt((position[0]-end[0])**2 + (position[1]-end[1])**2) #Euclidean Distance
    max_dist = np.sqrt((start[0]-end[0])**2 + (start[1]-end[1])**2) #Euclidean Distance
    norm_dist = (max_dist - distance_to_end)/max_dist #Normalise
    return norm_dist

In [57]:
def evaluate_heuristic_move(move,position, start,end,grid_size):
    position = move_position(grid_size,position,move,end)
    distance_to_end=np.sqrt((position[0]-end[0])**2 + (position[1]-end[1])**2) #Euclidean Distance
    max_dist = np.sqrt((start[0]-end[0])**2 + (start[1]-end[1])**2) #Euclidean Distance
    norm_dist = (max_dist - distance_to_end)/max_dist #Normalise
    return norm_dist

new_move

This function will return the best move based on the greedy policy(minimising distance to the obejctive) after analysing all the possible moves from the current position.

- grid_size: This will be in the form of a tuple (x,y) and is used to set the boundaries of the grid.
- position: This will be in the form of a tuple (x,y) and will be used as reference point from where to move from.
- moves: ["U", "D", "L", "R","UL", "DL", "UR","UL"] up & down, left & right and in diagonals.
- start: This will be in the form of a tuple (x,y).
- end: This will be in the form of a tuple (x,y).

In [58]:
def new_move(moves,start, end, grid_size,position):
    fitness = []
    for move in moves:
       total_distance = evaluate_move(grid_size,move,start,end,position) 
       fitness.append(total_distance)
    max_fitness = np.max(fitness)
    move = moves[fitness.index(max_fitness)]
    move = str(move).strip('[]')
    move = str(move).replace("'","")
    fitness= fitness[moves.index(move)]
    return move

In [59]:
def new_heuristic_move(moves, position,start, end, grid_size, move):
    fitness = []
    for move in moves:
        move_fitness = evaluate_heuristic_move(move,position, start, end, grid_size) 
        fitness.append(move_fitness)
    max_fitness = max(fitness)
    idx_max = fitness.index(max_fitness)
    move = moves[idx_max]
    move = str(move).strip('[]')
    move = str(move).replace("'","")
    return move

create_segment

This function will create a direct path between the start and end points of the segment.

- grid_size: This will be in the form of a tuple (x,y) and is used to set the boundaries of the grid.
- moves: ["U", "D", "L", "R","UL", "DL", "UR","UL"] up & down, left & right and in diagonals.
- start: This will be in the form of a tuple (x,y).
- end: This will be in the form of a tuple (x,y).

In [60]:
def create_segment(moves,start, end, grid_size):
    path = []
    position = start
    path.append(position)
    while position != end:
        move= new_move(moves, start, end, grid_size,position)
        position = tuple(move_position(grid_size,position,move,end))
        if position not in path:
            path.append(position)
    return path


create_path

This function will select a number of random points in the grid and create path segments joining them to go from the start to end position. The selection of random points will increase the exploration of the solution space in order to avoid getting stuck in local optima.

- grid_size: This will be in the form of a tuple (x,y) and is used to set the boundaries of the grid.
- moves: ["U", "D", "L", "R","UL", "DL", "UR","UL"] up & down, left & right and in diagonals.
- start: This will be in the form of a tuple (x,y).
- end: This will be in the form of a tuple (x,y).
- num_points: In the form of an integer from 0 to inf. A very high number of points will yield absurd and long to generate paths. Suggested: 2-6.

In [61]:
def create_path(start,end,moves, grid_size, num_points):
    int_points = []
    path = []
    int_points.append(start)
    i = start[0]
    j = start[1]
    for _ in range(num_points):
        i = random.randint(0,grid_size[0])
        j = random.randint(0,grid_size[1])
        position = (i,j)
        int_points.append(position)
    int_points.append(end)
    for x in range(int_points[0]):
        segment = create_segment(moves,int_points[x],int_points[x+1],grid_size)
        path.extend(segment)
    return path



In [62]:
def create_heuristic_path(moves,start, end, grid_size):
    path = []
    position = start
    path.append(position)
    move = 'R'
    while position != end:
        move= new_heuristic_move(moves, position,start, end, grid_size, move)
        position = tuple(move_position(grid_size,position,move,end))
        path.append(position)
    return path

In [63]:
def calc_dist(path):
    tot_distance=0
    for i in range((len(path)-1)):
        step1 = path[i]
        step2 = path[i+1]
        step_dist = np.sqrt((step1[0]-step2[0])**2 + (step1[1]-step2[1])**2)
        if step1[0] != step2[0] & step1[1]!=step2[1]:
            x=np.sqrt(2)
        else:
            x=1
        tot_distance = tot_distance + x*step_dist
    return tot_distance

In [64]:
def calc_slope(path,max_slope,slope_dem):
    slope = 0
    flag_max_slope = 0
    for i in range((len(path)-1)):
        position = path[i]
        slope_point = slope_dem[position[0]][position[1]]
        slope = slope + slope_point
        if slope_point >= max_slope:
            flag_max_slope = 1
    return slope,flag_max_slope

In [65]:
def calc_traverse_time(path,slope_dem):
    time = 0 
    for i in range((len(path)-1)):
        step1 = path[i]
        step2 = path[i+1]
        if step1[0] != step2[0] & step1[1]!=step2[1]:
            x=np.sqrt(2)
        else:
            x=1
        position = path[i]
        speed = 0.025 - (0.001*slope_dem[position[0]][position[1]])
        if speed <= 0:
            speed = 0.0005
        time_s = int(x/speed)
        time = time + time_s
    #time_total = str(datetime.timedelta(seconds=time))
    return time

In [66]:
def select_pop_max(pop, fitness,pop_size, num_to_select):
    selected =[]
    for _ in range(pop_size):
        while len(selected) != num_to_select:
            max_fitness = max(fitness)
            print(max_fitness)
            idx_max = fitness.index(max_fitness)
            fitness.pop(idx_max)
            if pop[idx_max] not in selected:
                selected.append(pop[idx_max])
            pop.pop(idx_max)
        return selected

In [67]:
def select_pop_rand(pop, fitness,pop_size, num_to_select):
    selected =[]
    for i in range(num_to_select):
        x = random.randint(0,(pop_size-i-1))
        fitness.pop(x)
        selected.append(pop[x])
        pop.pop(x)
    return selected

In [68]:
def crossover(parent1,parent2, moves, grid_size, crossover_rate=1):
    if random.random()< crossover_rate:
        x = random.randint(0,1)
        crossover_point1= random.randint(0,(len(parent1)-1))
        crossover_point2= random.randint(0,(len(parent2)-1))
        child = parent1[:crossover_point1] + create_path(parent1[crossover_point1],parent2[crossover_point2],moves, grid_size, x) + parent2[crossover_point2:]
        return child
    else:
        return parent1

In [69]:
def crossover2(parent1,parent2,crossover_rate=0.9):
    if random.random()< crossover_rate:
        common_point1 =[]
        common_point2 =[]
        for x in parent1:
            for y in parent2:
                if x == y:
                    common_point1 = tuple(x)
                    common_point2 = tuple(y)
        if  not common_point1 or common_point2:
            return parent1
        else:
            crossover_points= random.randint(0,len(common_point1))
            child = parent1[:parent1.index(common_point1[crossover_points])] + parent2[parent2.index(common_point2[crossover_points]):]
            return child
    else:
        return parent1

In [70]:
def crossover3(parent1,parent2,crossover_rate=0.9):
    if random.random()< crossover_rate:
            crossover_point= random.randint(0,(len(parent1)-1))
            child = parent1[:crossover_point] + parent2[crossover_point:]
            return child
    else:
        return parent1

In [71]:
def mutation(path, end, moves, grid_size, mutation_rate=0.5):
    if random.random()< mutation_rate:
            x = random.randint(1,2)
            mutation_point= random.randint(2,len(path[0]))
            start = path[mutation_point-2]
            path[:mutation_point].append(create_path(start,end,moves, grid_size,num_points=x))
    return path

In [72]:
def mutation2(path, mutation_rate=0.5):
    if random.random()< mutation_rate:
            mutation_point1= random.randint(0,(len(path)-1))
            mutation_point2= random.randint(0,(len(path)-1))
            place_holder = path[mutation_point1]
            path[mutation_point1] = path[mutation_point2]
            path[mutation_point2] = place_holder
    return path

termination_condition

This function will return if the condition for ending the loop has been reached.

- generations: The variable that will give the updated number of generations on that iteration.
- max_generations: In the form of an integer from 0 to inf).

In [73]:
def  termination_condition(generations, max_generations):
    return generations>= max_generations

In [74]:
def normalize_pop(pop,start,end,grid_size,pop_size,max_slope,slope_dem):
    tot_distance = []
    tot_slope = []
    tot_time = []
    flag_ms =np.ones(pop_size)
    min_dist = np.sqrt((start[0]-end[0])**2 + (start[1]-end[1])**2)
    max_tot_distance = grid_size[0]* grid_size[1]
    max_tot_slope =max_tot_distance * 90
    for i in range(len(pop)):
        tot_distance1 = calc_dist(pop[i])
        tot_slope1,flag= calc_slope(pop[i],max_slope,slope_dem)
        tot_time1 = calc_traverse_time(pop[i],slope_dem)
        flag_ms[i]=flag
        tot_distance.append(tot_distance1)
        tot_slope.append(tot_slope1)
        tot_time.append(tot_time1)
    tot_distance = np.array(tot_distance)
    tot_slope = np.array(tot_slope)
    tot_time = np.array(tot_time)
    norm_time = np.array((1-((tot_time-(min_dist/0.025))/tot_time)))
    norm_dist =np.array((1-(tot_distance-min_dist)/tot_distance))
    norm_slope = np.array((max_tot_slope-tot_slope)/max_tot_slope)
    #norm_fitness =np.add(norm_slope,norm_dist,3*norm_time)/5
    norm_fitness = norm_time
    norm_fitness = norm_fitness.tolist()    
    
    for i in range(pop_size):
        if flag_ms[i] == 1:
            norm_fitness[i] = norm_fitness[i]/2
    return norm_fitness,tot_slope,tot_distance

In [75]:
def normalize_path(path,start,end, grid_size, max_slope,slope_dem):
    tot_distance = []
    tot_slope = []
    flag=1
    min_dist = np.sqrt((start[0]-end[0])**2 + (start[1]-end[1])**2)
    max_tot_distance = grid_size[0]* grid_size[1]
    max_tot_slope =max_tot_distance * 90
    tot_distance = calc_dist(path)
    tot_slope,flag= calc_slope(path,max_slope,slope_dem)
    tot_time = calc_traverse_time(path,slope_dem)
    norm_time = (1-((tot_time-(min_dist/0.025))/tot_time))
    norm_dist =(1-(tot_distance-min_dist)/tot_distance)
    norm_slope = (max_tot_slope-tot_slope)/max_tot_slope
    #norm_fitness =(norm_slope+norm_dist+3*norm_time)/5
    norm_fitness = norm_time
    if flag == 1:
        norm_fitness = norm_fitness/2
    return norm_fitness

In [76]:
def deloop (path,grid_size,moves):
    n = random.randint(10,20)
    delooped_path = []
    section = []
    for i in range(0, len(path), n):
        if i+n >(len(path)-1):
            n = (len(path)) -i
        section = path[i:i+n]
        start = section[0]
        end = section[-1]
        min_dist = np.sqrt((start[0]-end[0])**2 + (start[1]-end[1])**2)
        s_dist = calc_dist(section)
        if  min_dist == 0:
            delooped_path.extend(section)
        else:
            if (s_dist/min_dist)>1.15:
                section = create_heuristic_path(moves,start, end, grid_size)
                delooped_path.extend(section) 
            else:
                delooped_path.extend(section)
    return delooped_path

In [77]:

def evolutionary_algorithm(grid_size,start,end,moves, pop, pop_size, max_generations, deloop_flag,norm_fitness,max_slope,slope_dem):
    generations = 0
    fitnesses_gen = []
    while not termination_condition(generations,max_generations):
        t1_start = perf_counter() 
        generations += 1

        #BUILD NEXT GEN
        next_gen = []  

        #SELECT POPULATION FOR NEXT GEN
        selected_pop_size = int(pop_size/8)
        select_fitness = copy.deepcopy(norm_fitness)
        pop_w_elite = copy.deepcopy(pop)
        elite_pop = select_pop_max(pop,select_fitness, pop_size, selected_pop_size*2)
        select_fitness = copy.deepcopy(norm_fitness)
        selected_pop = select_pop_rand(pop_w_elite,select_fitness, pop_size, selected_pop_size*2)
        selected_pop = selected_pop +  elite_pop
        

        #ELITE
        for i in range(selected_pop_size*2):
            next_gen.append(elite_pop[i])

        pop_indexes = list(np.arange(0,(selected_pop_size*4),1,dtype=int))

        #GENERAL    
        for i in range(0, (selected_pop_size*4),2):
            i,j = random.sample(pop_indexes,k=2)
            pop_indexes.remove(i)
            pop_indexes.remove(j)
            parent1 =[]
            parent2=[]
            child1=[]
            child2=[]
            parent1 , parent2 = selected_pop[i],selected_pop[j]
            child1,child2 = crossover(parent1,parent2, moves, grid_size), crossover(parent2,parent1, moves, grid_size)
            #MUTATION
            child1,child2 = mutation(child1,end, moves, grid_size),mutation(child2,end, moves, grid_size)
            next_gen.append(child1)
            next_gen.append(child2)

        #IMIGRANT POPULATION
        for i in range(selected_pop_size*2):
            x=random.randint(1,3)
            imigrant = create_path(start,end,moves, grid_size,num_points=x)
            next_gen.append(imigrant)
        if len(next_gen) != pop_size:
            print("Next GEN Size different than Pop Size")
            break  
        
        pop[:] = next_gen

        #EVALUATE POPULATION
        norm_fitness = []
        norm_fitness,tot_slope,tot_distance = normalize_pop(pop,start,end,grid_size,pop_size,max_slope,slope_dem)
        if norm_fitness == [0]*len(norm_fitness):
            print('Norm_fitness is full of zeros')
            break 

        #DELOOP
        for i in range(pop_size):
            if deloop_flag == True:
                delooped_path = deloop(pop[i],grid_size,moves)
                delooped_fitness = normalize_path(delooped_path,start,end, grid_size, max_slope,slope_dem)
                if norm_fitness[i] <= delooped_fitness:
                    pop[i] = delooped_path
                    norm_fitness[i] = delooped_fitness

        #EVALUATE POPULATION
        norm_fitness = []
        norm_fitness,tot_slope,tot_distance = normalize_pop(pop,start,end,grid_size,pop_size,max_slope,slope_dem)
        for i in range(pop_size):
            if norm_fitness == [0]*len(norm_fitness):
                print('Norm_fitness is full of zeros')
                break 

        #GENERATION INFO
        best_fitness = max(norm_fitness)
        fitnesses_gen.append(best_fitness)
        best_path_idx = norm_fitness.index(best_fitness)
        print(f'Generation {generations}: Best fitness = {best_fitness}.')
        t1_stop = perf_counter()
        gen_compt = t1_stop-t1_start
    best_path = pop[best_path_idx]
    best_t_dist = tot_distance[best_path_idx]
    best_t_slp =  tot_slope[best_path_idx]
    print('Total distance:',best_t_dist)
    print('Total slope:',best_t_slp)
    return best_path, fitnesses_gen, gen_compt


In [78]:
def ga(start,end):
    t1_start = perf_counter() 

    #DEM 

    filename = "data/dems/lunargem_4to5_slope.tif"
    dem = rio.open(filename)
    print(filename)
    slope_dem =dem.read(1)

    # SET GRID SIZE
    grid_size=(dem.shape[0],dem.shape[1])
    moves=["U", "D", "L", "R","UL", "DL", "UR","UL"]

    #SYSTEM CONSTRAINTS
    max_slope = 15

    # INITIALIZE POPULATION(DIFFERENT RANDOM PATHS)
    pop_size= 32 #Always multiples of 8!
    print("Population size =", pop_size)
    pop = []
    max_generations= 10
    deloop_flag = True


    for i in range(pop_size):
        x=random.randint(1,4)
        populant = create_path(start,end,moves, grid_size, num_points=x)
        pop.append(populant)

    #EVALUATE POPULATION
    norm_fitness = []
    norm_fitness,tot_slope,tot_distance = normalize_pop(pop,start,end,grid_size,pop_size,max_slope,slope_dem)
    for i in range(pop_size):
        if deloop_flag == True:
            delooped_path = deloop(pop[i],grid_size,moves)
            delooped_fitness = normalize_path(delooped_path,start,end, grid_size, max_slope,slope_dem)
            if norm_fitness[i] <= delooped_fitness:
                pop[i] = delooped_path
                norm_fitness[i] = delooped_fitness

    #GENERATION INFO
    best_fitness = max(norm_fitness)
    best_path_idx = norm_fitness.index(best_fitness)
    print(f'Generation 0: Best fitness = {best_fitness}.')
    
    # RUN EVOLUTIONARY ALGORITHM

    best_path,fitnesses_gen = evolutionary_algorithm(grid_size,start,end,moves, pop, pop_size, max_generations, deloop_flag,norm_fitness,max_slope,slope_dem)
    t1_stop = perf_counter()

    print('Best path found:',best_path)
    print("Elapsed time during the whole program in minutes:", (t1_stop-t1_start)/60)
    #traverse_time = calc_traverse_time(best_path)
    time_total = calc_traverse_time(best_path,slope_dem)
    traverse_time = str(datetime.timedelta(seconds=time_total))
    print ('Total traverse time:',traverse_time)
    return best_fitness,fitnesses_gen,traverse_time